## Calibrate Insulation Market Share

**Purpose**
Uses iterative proportional fitting to reconcile insulation measure shares and produce calibration-ready weights.

**Primary inputs**
- `data/market_share/insulation.csv`
- `data/market_share/market_share_insulation_tremi.csv`

**Primary outputs**
- `data/market_share/market_share_insulation_cee.csv`



## Execution Notes

- Run cells top-to-bottom from a clean kernel.
- Paths are notebook-local and follow the refactored domain layout (`data/` for inputs and small derived tables, `runs/` for generated scenario outputs).
- If you switch datasets or scenarios, update the path/config variables in the first code cells before running downstream steps.



## Iterative Proportional fitting


In [ ]:
import os
import pandas as pd
import numpy as np



In [ ]:
data_tremi = pd.read_csv(os.path.join('data', 'market_share', 'insulation.csv'), index_col=[0, 1, 2, 3]).squeeze()
data_tremi


In [ ]:
market_share = pd.read_csv(os.path.join('data', 'market_share', 'market_share_insulation_tremi.csv'), index_col=[0, 1, 2, 3]).squeeze().rename(None)
market_share


In [ ]:
ms_cee = dict()
ms_cee['Wall'] = pd.Series([0.1021, 1 - 0.1021], index=pd.Index([True, False], name='Wall')) * 1_000
ms_cee['Floor'] = pd.Series([0.3038, 1 - 0.3038], index=pd.Index([True, False], name='Floor')) * 1_000
ms_cee['Roof'] = pd.Series([0.4972, 1 - 0.4972], index=pd.Index([True, False], name='Roof')) * 1_000
ms_cee['Windows'] = pd.Series([0.0969, 1 - 0.0969], index=pd.Index([True, False], name='Windows')) * 1_000
ms_cee


In [ ]:
def ipf_update(M, ms_cee):
    for i in M.index.names:
        M = ms_cee[i] * M / M.groupby(i).sum()

    d = {i: np.linalg.norm(ms_cee[i] - M.groupby(i).sum(), 2) for i in M.index.names}
    return M, d


In [ ]:
M = data_tremi.copy()
for _ in range(10):
    M, d = ipf_update(M, ms_cee)
    print(d)


In [ ]:
weight_m = M / M.sum()
weight_m


In [ ]:
pd.set_option('display.float_format', lambda x: '%.6f' % x)
df = pd.concat((weight_m, market_share), axis=1)
df


In [ ]:
weight_m.to_csv(os.path.join('data', 'market_share', 'market_share_insulation_cee.csv'))


## Ad hoc method

We calibrate only number of work with the technology


In [ ]:
from scipy.optimize import fsolve
from project.utils import reindex_mi


In [ ]:
def func(coeff, ms, target):
    names = ['Wall', 'Floor', 'Roof', 'Windows']
    coeff = pd.Series(coeff, index=pd.MultiIndex.from_tuples(((True, False, False, False),
                                                              (False, True, False, False),
                                                              (False, False, True, False),
                                                              (False, False, False, True),
                                                                 ), names=names))
    ms = reindex_mi(coeff, ms.index).fillna(1) * ms

    return np.array([ms.xs(True, level='Wall').sum() / ms.sum() - target[i] for i in names])






In [ ]:
names = ['Wall', 'Floor', 'Roof', 'Windows']
coeff = pd.Series(coeff, index=pd.MultiIndex.from_tuples(((True, False, False, False),
                                                          (False, True, False, False),
                                                          (False, False, True, False),
                                                          (False, False, False, True),
                                                             ), names=names))
ms = reindex_mi(coeff, ms.index).fillna(1) * ms


In [ ]:
func(x0, ms, target)


In [ ]:
x0 = np.array([1, 1, 1, 1])
ms = market_share.copy()
target = {'Wall': 0.1021, 'Floor': 0.3038, 'Roof': 0.4972, 'Windows':0.0969 }

